<a href="https://colab.research.google.com/github/dJasawat/Guvi_Assignmnets5_AI-Powered-Banking-Support-Fraud-Intelligence-System-using-NLP-RAG/blob/main/AI_Banking_RAG_Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
%pip -q install sentence-transformers groq qdrant-client

In [ ]:
import os

In [42]:
import os

if not os.path.isdir("RAG_Data"):
    from google.colab import files
    uploaded = files.upload()          # select RAG_Data.zip
    !unzip -o -q RAG_Data.zip

print("Files in RAG_Data/:")
print(sorted(os.listdir("RAG_Data")))

Saving RAG_Data.zip to RAG_Data.zip
Files in RAG_Data/:
['04_qa_pairs.json', 'fraud_handling_policy.txt', 'kyc_policy.txt', 'loan_processing_policy.txt', 'refund_dispute_policy.txt']


In [3]:
import getpass

os.environ["GROQ_API_KEY"]= getpass.getpass("Paste your Groq key: ")
print("Key saved for this session ✔")

Paste your Groq key: ··········
Key saved for this session ✔


In [4]:
from groq import Groq
from sentence_transformers import SentenceTransformer

# it reads groq key automatically
groq_client = Groq()

GROQ_MODEL="openai/gpt-oss-120b"

# Downloads the model the first time (~90 MB), then it's cached
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Connected to Groq and embedding model loaded ✔")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Connected to Groq and embedding model loaded ✔


In [43]:
from pathlib import Path

RAG_Data_Folder= Path("RAG_Data")
documents={}

for file in sorted(RAG_Data_Folder.glob("*.txt")):
  documents[file.name]=file.read_text(encoding="utf-8")

print(f"Loaded {len(documents)} documents:\n")
for name,text in documents.items():
  print(f"{name:35s} {len(text):>7,} characters")


Loaded 4 documents:

fraud_handling_policy.txt             3,236 characters
kyc_policy.txt                        2,516 characters
loan_processing_policy.txt            2,959 characters
refund_dispute_policy.txt             1,994 characters


In [6]:
# split Texts
def split_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    text = " ".join((text or "").split())  #it's a text-cleaning operation. It removes unnecessary whitespace from a string and replaces multiple spaces/newlines/tabs with a single space.

    if not text:
        return []
    if chunk_size < 50:
        raise ValueError("Chunk size should be at least 100 characters.")
    if overlap < 0 or overlap >= chunk_size:
        raise ValueError("Overlap must be zero or smaller than chunk size.")

    chunks = []
    start = 0

    while start < len(text):
        end = min(start + chunk_size, len(text))
        chunk = text[start:end]

        if end < len(text):
            preferred_break = max(
                chunk.rfind(". "),
                chunk.rfind("? "),
                chunk.rfind("! "),
                chunk.rfind("\n"),
            )
            if preferred_break >= int(chunk_size * 0.55):
                end = start + preferred_break + 1
                chunk = text[start:end]

        chunk = chunk.strip()
        if chunk:
            chunks.append(chunk)

        if end >= len(text):
            break

        start = max(end - overlap, start + 1)

    return chunks


In [7]:
all_chunks=[]

for filename,text in documents.items():
  for i,chunk in enumerate(split_text(text)):
    all_chunks.append(
        {"id":f"{filename}_chunk_{i}",
         "text":chunk,
         "source":filename,
        })
print(f"Total chunks across all documents: {len(all_chunks)}")
print("\nExample chunk:")
print(all_chunks[5])

Total chunks across all documents: 49

Example chunk:
{'id': 'fraud_handling_policy.txt_chunk_5', 'text': 'r transaction amount, subject to investigation. 4. BANK RESPONSE SLA 4.1 Acknowledgement: Within 1 working hour of complaint. 4.2 Card/Account Block: Immediately on customer request, within 30 minutes. 4.3 Provisional Credit: Within 10 working days for CNP fraud.', 'source': 'fraud_handling_policy.txt'}


In [ ]:
# Generate QA Pair text , encode them  and save into quadrant
import pandas as pd
qaDoc= pd.read_json("/content/RAG_Data/04_qa_pairs.json")
qaDoc

# Extract questions and answers
questions = qaDoc["question"].tolist()
answers = qaDoc["answer"].tolist()

# Encode the questions and answers
print("\nEncoding questions...")
question_vectors = embedding_model.encode(questions)
print("Encoding answers...")
answer_vectors = embedding_model.encode(answers)

print(f"Encoded {len(question_vectors)} questions with dimension {question_vectors.shape[1]}")
print(f"Encoded {len(answer_vectors)} answers with dimension {answer_vectors.shape[1]}")

# Store encoded QA pairs with their original data
encoded_qa_pairs = []
for i, row in qaDoc.iterrows():
    encoded_qa_pairs.append({
        "id": row["id"],
        "category": row["category"],
        "question": row["question"],
        "answer": row["answer"],
        "suggested_action": row["suggested_action"],
        "question_vector": question_vectors[i].tolist(), # Convert numpy array to list for storage
        "answer_vector": answer_vectors[i].tolist()
    })

print("\nExample of an encoded QA pair:")
if encoded_qa_pairs:
    example_encoded_qa = encoded_qa_pairs[0]
    print(f"Question ID: {example_encoded_qa['id']}")
    print(f"Question: {example_encoded_qa['question']}")
    print(f"Answer: {example_encoded_qa['answer']}")
    print(f"Question Vector (first 5 elements): {example_encoded_qa['question_vector'][:5]}")
    print(f"Answer Vector (first 5 elements): {example_encoded_qa['answer_vector'][:5]}")

In [8]:
#Text Embeddings
import numpy as np
import pandas as pd

def embed_texts(texts: list[str]) -> list[list[float]]:
    cleaned = [(text or "").strip() for text in texts]
    if not cleaned or any(not text for text in cleaned):
        raise ValueError("Every text must contain content.")
    return [list(vector) for vector in embedding_model.encode(cleaned)]

vectors = embed_texts([c["text"] for c in all_chunks])
vectors= np.array(vectors).astype("float32")

dimension = vectors.shape[1]

In [9]:
print(f"Vector length: {len(vectors)}")
print(f"Vector dimension: {dimension}")
print(f"First 8 numbers: {[[round(x,4) for x in v[:5]] for v in vectors[:8]]}")

Vector length: 49
Vector dimension: 384
First 8 numbers: [[np.float32(-0.0478), np.float32(0.031), np.float32(-0.0178), np.float32(-0.0787), np.float32(0.024)], [np.float32(-0.0009), np.float32(0.0535), np.float32(-0.0214), np.float32(0.0107), np.float32(0.0741)], [np.float32(-0.063), np.float32(0.0393), np.float32(-0.0101), np.float32(-0.0766), np.float32(0.0223)], [np.float32(-0.0105), np.float32(0.027), np.float32(-0.0598), np.float32(-0.08), np.float32(-0.0358)], [np.float32(-0.022), np.float32(0.0494), np.float32(-0.0512), np.float32(-0.0323), np.float32(-0.0303)], [np.float32(-0.1159), np.float32(0.0241), np.float32(-0.0425), np.float32(-0.0414), np.float32(0.0145)], [np.float32(-0.1553), np.float32(0.0288), np.float32(0.054), np.float32(-0.0281), np.float32(0.0496)], [np.float32(-0.0159), np.float32(0.0329), np.float32(-0.0475), np.float32(0.0025), np.float32(0.024)]]


In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Embed the sample query
query_vector = embedding_model.encode([sample_query])[0].reshape(1, -1)

# Calculate cosine similarity between the query vector and all chunk vectors
# 'vectors' contains the embeddings of all chunks, each row is a chunk's embedding
similarity_scores = cosine_similarity(query_vector, vectors)

# The result is a 2D array, we need the first (and only) row
similarity_scores = similarity_scores[0]

# Calculate the average similarity score
average_similarity = similarity_scores.mean()

print(f"Average similarity score for all chunks with the query '{sample_query}': {average_similarity:.4f}")

# Optionally, you can also see the min and max scores for context
print(f"Minimum similarity score: {similarity_scores.min():.4f}")
print(f"Maximum similarity score: {similarity_scores.max():.4f}")

NameError: name 'sample_query' is not defined

In [11]:

from qdrant_client import QdrantClient, models
os.environ["QDRANT_URL"] = "https://f2562d30-a491-4781-b7b1-0fe3bec4f6fd.sa-east-1-0.aws.cloud.qdrant.io"
os.environ["QDRANT_API_KEY"] = getpass.getpass("Paste your Qdrant key: ")
print("Key saved for this session ✔")

Paste your Qdrant key: ··········
Key saved for this session ✔


In [25]:
client=QdrantClient(
    url=os.environ["QDRANT_URL"],
    api_key=os.environ["QDRANT_API_KEY"]
)
COLLECTION_NAME="RAG_Data"

In [13]:
print(client.get_collections())

collections=[CollectionDescription(name='RAG_Data')]


In [ ]:
def upsert_vectors_to_qdrant(qdrant_url: str, qdrant_api_key: str, collection_name: str, vectors: np.ndarray, all_chunks: list):
    try:
        # Check if collection exists, create if not
        if not client.collection_exists(collection_name=collection_name):
            client.create_collection(
                collection_name=collection_name,
                vectors_config=models.VectorParams(size=vectors.shape[1], distance=models.Distance.COSINE),
            )
            print(f"Collection '{collection_name}' created.")
        else:
            print(f"Collection '{collection_name}' already exists.")

    except Exception as e:
        print(f"Error creating or checking collection '{collection_name}': {e}")
        return

    # Prepare points for upsert
    points = []
    for i, vector in enumerate(vectors):
        payload = {
            "text": all_chunks[i]["text"],
            "source": all_chunks[i]["source"],
            "id": all_chunks[i]["id"]
        }
        points.append(
            models.PointStruct(
                id=i,
                vector=vector.tolist(), # Qdrant expects list, not np.ndarray
                payload=payload
            )
        )

    try:
        # Upsert points
        operation_info = client.upsert(
            collection_name=collection_name,
            wait=True,
            points=points,
        )
        print(f"Upsert operation info: {operation_info}")
        print(f"Successfully saved {len(points)} embeddings to Qdrant collection '{collection_name}'.")

    except Exception as e:
        print(f"Error upserting vectors to collection '{collection_name}': {e}")

In [ ]:
# Save embedings in quadrants
#collection_name = "RAG_Data"
upsert_vectors_to_qdrant(os.environ["QDRANT_URL"], os.environ["QDRANT_API_KEY"], COLLECTION_NAME, vectors, all_chunks)


Collection 'RAG_Data' already exists.
Upsert operation info: operation_id=2 status=<UpdateStatus.COMPLETED: 'completed'>
Successfully saved 49 embeddings to Qdrant collection 'RAG_Data'.


### Search Function for Qdrant

This function will embed a given query and use it to search the Qdrant collection for the most relevant documents.

In [15]:
def search_qdrant(query: str, collection_name: str, embedding_model, client, top_k: int = 5) -> list[dict]:
    # Embed the query
    query_vector = embedding_model.encode([query])[0].tolist()

    # Perform the search
    search_result = client.query_points(
        collection_name=collection_name,
        query=query_vector,
        limit=top_k,
        with_payload=True # Include payload in the results
    )

    # Extract relevant information from the results
    formatted_results = []
    for point in search_result.points:
        payload = point.payload or {}
        formatted_results.append({
            "score": point.score,
            "text": payload.get("text", ""),
            "source": payload.get("source", ""),
            "id": payload.get("id", "")
        })

    return formatted_results

In [39]:
def format_recent_history(history: list, max_messages: int = 6) -> str:
    if not history:
        return "No earlier conversation."

    lines = []
    for item in history[-max_messages:]:
        if isinstance(item, dict):
            role = item.get("role", "user")
            content = item.get("content", "")
            lines.append(f"{role.title()}: {content}")
        elif isinstance(item, (list, tuple)) and len(item) == 2:
            lines.append(f"User: {item[0]}")
            lines.append(f"Assistant: {item[1]}")

    return "\n".join(lines) or "No earlier conversation."

def rag_chat(
    message: str,
    history: list,
    qdrant_url: str,
    qdrant_api_key: str,
    collection: str,
    groq_api_key: str,
    model_name: str,
    top_k: int,
    embedding_model, # New parameter
    client,          # New parameter
):
    history = list(history or [])

    try:
        message = (message or "").strip()
        if not message:
            return "", history

        # Corrected call to search_qdrant
        contexts = search_qdrant(
            message,
            collection,
            embedding_model, # Passed as parameter
            client,          # Passed as parameter
            top_k,
        )

        if not contexts:
            answer = "I could not find relevant information in the stored documents."
        elif not (groq_api_key or "").strip():
            answer = (
                "Retrieval succeeded, but no Groq key was provided. "
                "Here are the retrieved chunks:\n\n"
                + "\n\n".join(contexts)
            )
        else:
            # Extract text from each context dictionary for joining
            context_texts = [c["text"] for c in contexts]

            # Buid Prompt
            prompt = f"""
            You are a careful retrieval-augmented assistant.

            Answer the current question using only the supplied context.
            Use recent conversation only to understand pronouns or follow-up questions.
            Do not introduce facts that are absent from the context.
            When the context does not contain the answer, say:
            "I could not find this information in the uploaded documents."
            Cite relevant source numbers such as [Source 1].

             Formatting rules:
            - Give a clear, concise answer.
            - Use Markdown formatting.
            - Use a short heading when appropriate.
            - Use bullet points for multiple items.
            - Use numbered lists for procedures or steps.
            - Use **bold** only for important terms.
            - Keep paragraphs short.
            - Do not repeat the conversation history.
            - Do not mention information that is not supported by the context.
            - Put sources at the end of the answer.
            - Do not use HTML.


            Current question:{message}
            Recent conversation:{format_recent_history(history)}

            Retrieved context:{chr(10).join(context_texts)}""".strip()

            # Call LLM
            groq_client = Groq(api_key=groq_api_key.strip())
            completion = groq_client.chat.completions.create(
                model=(model_name or "openai/gpt-oss-20b").strip(),
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "You answer document questions accurately and "
                            "refuse to invent unsupported information."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                temperature=0.2,
                max_completion_tokens=1200,
            )
            answer = completion.choices[0].message.content

        history.extend(
            [
                {"role": "user", "content": message},
                {"role": "assistant", "content": answer},
            ]
        )
        return answer

    except Exception as exc:
        answer = f"❌ {exc}"
        history.extend(
            [
                {"role": "user", "content": message},
                {"role": "assistant", "content": answer},
            ]
        )
        return  answer

def clear_chat():
    return []

rag_chat("What is the fraud handling policy?","RAG_Data",
         os.environ["QDRANT_URL"],
         os.environ["QDRANT_API_KEY"],
         COLLECTION_NAME,
         os.environ["GROQ_API_KEY"],
         GROQ_MODEL,
         3,
         embedding_model, # Passed as parameter
         client
         )


'### Fraud Handling Policy (Retail Banking)\n\n- **Scope**: Applies to all unauthorized transactions, including:\n  - Card fraud  \n  - UPI fraud  \n  - Net‑banking fraud  \n  - Identity‑theft cases reported by retail banking customers  \n\n- **Issuing Authority**: Chief Risk Officer  \n- **Version**: 3.2, effective 01‑Jan‑2024  \n\n*Source: [FRAUD HANDLING POLICY — RETAIL BANKING, Version 3.2]*'

In [38]:
import os
import gradio as gr
import traceback


def gradio_rag_chat(message, history):
    try:
        # Get API credentials
        qdrant_url = os.environ["QDRANT_URL"]
        qdrant_api_key = os.environ["QDRANT_API_KEY"]
        groq_api_key = os.environ["GROQ_API_KEY"]

        # Model configuration
        model_name = GROQ_MODEL
        top_k = 3

        # Call RAG pipeline
        answer = rag_chat(
            message,
            history,
            qdrant_url,
            qdrant_api_key,
            COLLECTION_NAME,
            groq_api_key,
            model_name,
            top_k,
            embedding_model,
            client
        )

        # ChatInterface expects the assistant response
        return answer

    except Exception as e:

        print("\n--- GRADIO DEBUG ERROR ---")
        traceback.print_exc()
        print(f"Error in gradio_rag_chat: {e}")

        # Return only the error message
        return f"❌ An internal error occurred: {e}"


demo = gr.ChatInterface(
    fn=gradio_rag_chat,
    examples=[
        "What is the fraud handling policy?",
        "What are the steps for processing a loan?",
        "Summarize the KYC policy."
    ],
    title="RAG Chatbot"
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d789f9ca3a6c7d93b0.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### Demo of the Search Function

Let's test the search function with a sample query.

In [ ]:
sample_query = "What are the steps for processing a loan?"
search_results = search_qdrant(sample_query, collection_name="RAG_Data", embedding_model=embedding_model, top_k=3)

print(f"Search results for query: '{sample_query}'\n")
for i, result in enumerate(search_results):
    print(f"--- Result {i+1} (Score: {result['score']:.4f}) ---")
    print(f"Source: {result['source']}")
    print(f"ID: {result['id']}")
    print(f"Text: {result['text']}\n")

TypeError: search_qdrant() missing 1 required positional argument: 'client'